# Task 1: Term Deposit Subscription Prediction (Bank Marketing)
**DevelopersHub Corporation – Data Science & Analytics Advanced Internship**

---

## Problem Statement
Predict whether a bank customer will subscribe to a term deposit as a result of a marketing campaign.

## Objective
- Load and explore the Bank Marketing Dataset (UCI)
- Encode all categorical features properly
- Train classification models: Logistic Regression & Random Forest
- Evaluate using Confusion Matrix, F1-Score, and ROC Curve
- Use SHAP to explain at least 5 model predictions

## Dataset
**Source:** UCI Machine Learning Repository – Bank Marketing Dataset  
**Target Column:** `y` → whether client subscribed to a term deposit (yes/no)

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, roc_curve, auc, ConfusionMatrixDisplay
)
import shap

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All libraries imported successfully!")

ModuleNotFoundError: No module named 'shap'

## 2. Load Dataset

In [ ]:
# Download and load the dataset directly from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

import urllib.request
import zipfile
import io

print("Downloading dataset...")
response = urllib.request.urlopen(url)
zip_data = zipfile.ZipFile(io.BytesIO(response.read()))

# Use the full dataset
with zip_data.open('bank-additional/bank-additional-full.csv') as f:
    df = pd.read_csv(f, sep=';')

print(f"✅ Dataset loaded! Shape: {df.shape}")
df.head()

## 3. Dataset Description & Exploration

In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Rows   : {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print()
print("Column Info:")
df.info()

In [ ]:
# Statistical summary
df.describe(include='all')

In [ ]:
# Check for missing values
print("Missing Values per Column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "✅ No missing values found!")

In [ ]:
# Target distribution
print("Target Variable Distribution:")
print(df['y'].value_counts())
print()
print(df['y'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['y'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#4C72B0', '#DD8452'], edgecolor='black')
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subscribed (y)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Subscription Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features vs target
cat_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['y'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#4C72B0', '#DD8452'], edgecolor='black')
    axes[i].set_title(f'{col.capitalize()} vs Subscription Rate', fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].legend(['No', 'Yes'], title='Subscribed')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Features vs Target', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical features distribution
num_cols = ['age', 'duration', 'campaign', 'pdays', 'previous',
            'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df.groupby('y')[col].plot(kind='kde', ax=axes[i], legend=True)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Numerical Features – Distribution by Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
plt.figure(figsize=(12, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix – Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing

In [ ]:
# Work on a copy
df_processed = df.copy()

# Encode target variable
df_processed['y'] = df_processed['y'].map({'yes': 1, 'no': 0})

# Identify categorical columns
categorical_cols = df_processed.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", categorical_cols)

In [ ]:
# Label encode all categorical features
le = LabelEncoder()
label_encoders = {}

for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"  Encoded: {col} → {df[col].unique()[:4]}...")

print()
print("✅ All categorical features encoded!")
df_processed.head(3)

In [ ]:
# Split features and target
X = df_processed.drop('y', axis=1)
y = df_processed['y']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape : {y.shape}")
print(f"Class balance       : {y.value_counts().to_dict()}")

In [ ]:
# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")

## 6. Model Training

In [ ]:
# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred  = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Logistic Regression trained!")

In [ ]:
# --- Model 2: Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=15,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("✅ Random Forest trained!")

## 7. Model Evaluation

In [ ]:
def print_report(name, y_true, y_pred):
    print(f"{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(classification_report(y_true, y_pred, target_names=['No', 'Yes']))
    print(f"F1 Score (macro): {f1_score(y_true, y_pred, average='macro'):.4f}")

print_report("Logistic Regression", y_test, lr_pred)
print_report("Random Forest",       y_test, rf_pred)

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, pred in zip(
    axes,
    ['Logistic Regression', 'Random Forest'],
    [lr_pred, rf_pred]
):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for name, proba in [('Logistic Regression', lr_proba), ('Random Forest', rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance – Random Forest
feature_importance = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feature_importance.head(15).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances – Random Forest', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Explainable AI with SHAP

In [ ]:
# Initialize SHAP TreeExplainer for Random Forest
print("Initializing SHAP explainer (this may take a moment)...")
explainer = shap.TreeExplainer(rf_model)

# Compute SHAP values for the test set (use a sample for speed)
X_test_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(X_test_sample)

# shap_values[1] = SHAP values for class 1 (subscribed = yes)
print("✅ SHAP values computed!")

In [ ]:
# SHAP Summary Plot – Beeswarm (Global Feature Impact)
plt.figure()
shap.summary_plot(
    shap_values[1], X_test_sample,
    plot_type='dot', show=False, max_display=15
)
plt.title('SHAP Summary Plot – Random Forest (Class: Subscribed)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot – Mean Absolute Impact
plt.figure()
shap.summary_plot(
    shap_values[1], X_test_sample,
    plot_type='bar', show=False, max_display=15
)
plt.title('SHAP Mean Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Waterfall Plots – Explain 5 Individual Predictions
print("Explaining 5 individual predictions with SHAP waterfall plots...\n")

# Use SHAP Explanation object for waterfall plots
shap_explanation = shap.TreeExplainer(rf_model)(X_test_sample)

for i in range(5):
    actual    = y_test.iloc[i]
    predicted = rf_model.predict(X_test_sample.iloc[[i]])[0]
    prob      = rf_model.predict_proba(X_test_sample.iloc[[i]])[0][1]

    print(f"--- Prediction {i+1} ---")
    print(f"  Actual    : {'Subscribed ✅' if actual == 1 else 'Not Subscribed ❌'}")
    print(f"  Predicted : {'Subscribed ✅' if predicted == 1 else 'Not Subscribed ❌'}")
    print(f"  Probability of Subscription: {prob:.2%}")

    plt.figure()
    shap.waterfall_plot(shap_explanation[i], max_display=12, show=False)
    plt.title(f"SHAP Waterfall – Prediction {i+1} | Actual: {'Yes' if actual==1 else 'No'} | Pred: {'Yes' if predicted==1 else 'No'}",
              fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_pred_{i+1}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## 9. Model Comparison Summary

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy':  [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred)
    ],
    'Precision': [
        precision_score(y_test, lr_pred),
        precision_score(y_test, rf_pred)
    ],
    'Recall':    [
        recall_score(y_test, lr_pred),
        recall_score(y_test, rf_pred)
    ],
    'F1 Score':  [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred)
    ],
    'ROC AUC': [
        auc(lr_fpr, lr_tpr),
        auc(rf_fpr, rf_tpr)
    ]
})

summary = summary.set_index('Model').round(4)
print("Model Comparison Summary:")
summary.style.highlight_max(color='lightgreen')

## 10. Conclusion & Insights

### Key Findings

1. **Class Imbalance:** The dataset is heavily imbalanced (~88% No, ~12% Yes). We used `class_weight='balanced'` to handle this.

2. **Best Performing Model:** Random Forest outperformed Logistic Regression on AUC and F1-score, demonstrating its ability to capture non-linear relationships.

3. **Most Important Features (from SHAP):**
   - `duration` – Call duration is the strongest predictor. Longer calls strongly indicate subscription.
   - `euribor3m` – Euribor 3-month rate (economic indicator) is highly influential.
   - `nr.employed` – Number of employees (macroeconomic factor) matters significantly.
   - `poutcome` – Previous campaign outcome is very predictive.
   - `age` – Older customers tend to subscribe more.

4. **Business Insight:**
   - Focus marketing calls on customers with prior positive campaign contact (`poutcome = success`).
   - Longer engagement during calls significantly increases subscription probability.
   - Economic conditions (euribor rate, employment numbers) heavily influence customer decisions.

5. **XAI with SHAP:** Waterfall plots reveal the exact contribution of each feature for individual predictions, making the model transparent and interpretable for business stakeholders.